# SciGeoGuard-X — Publication Experiments v2
## Real EO pixels, diverse SilentGeoBench cases, held-out error families, and stronger evaluation

This notebook is the **next-stage experiment notebook** for SciGeoGuard-X.

It is deliberately different from the first prototype:

1. Adds live remote access hooks for **real Sentinel-2 L2A COGs**, **ESA WorldCover**, **Copernicus DEM GLO-30**, and **NASA GPM IMERG V07**.
2. Adds real pixel/window experiments where network access is available.
3. Expands SilentGeoBench from repeated templates into **parameter-diverse cases across many error families**.
4. Separates **development error families** from **held-out test families**.
5. Adds stronger baselines and publication metrics.
6. Exports benchmark tables for review.

> Important: the generated benchmark in this notebook is **not claimed to be expert-reviewed**. It is an automatically generated stress-test set that should later be reviewed by domain experts before a publication makes such a claim.

## 0. Execution mode

The notebook is safe to execute in two modes:

- `LIVE_DATA = False` — fully runs using real product specifications + deterministic local fixtures.
- `LIVE_DATA = True` — additionally reads real pixels/subsets from public EO endpoints in Colab.

The saved notebook defaults to offline-safe mode so every core experiment can execute without authentication.

In [ ]:
LIVE_DATA = False
SEED = 42

from dataclasses import dataclass, field, asdict, replace
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from datetime import datetime, timezone, timedelta
import hashlib, json, math, random, itertools, statistics

import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

print("SciGeoGuard-X v2 ready.")
print("LIVE_DATA =", LIVE_DATA)

# Part I — Real Earth-observation sources

The endpoints below correspond to actual public Earth-observation products.

### Sentinel-2 L2A
Real Planetary Computer STAC item:

`S2A_MSIL2A_20231107T144731_R139_T20QRF_20231107T182159`

### ESA WorldCover
A public mirror of the official ESA WorldCover 2021 v200 tile:

`ESA_WorldCover_10m_2021_v200_N18W063_Map.tif`

### Copernicus DEM
Public AWS GLO-30 COG tile pattern.

### NASA GPM IMERG V07
Public NOAA ERDDAP mirror of NASA IMERG Final Run V07.

In [ ]:
SOURCES = {
    "sentinel2": {
        "item_id": "S2A_MSIL2A_20231107T144731_R139_T20QRF_20231107T182159",
        "epsg": 32620,
        "datetime": "2023-11-07T14:47:31.024000Z",
        "cloud_cover_percent": 4.217,
        "B04": {
            "role": "red", "gsd_m": 10,
            "url": "https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/20/Q/RF/2023/11/07/"
                   "S2A_MSIL2A_20231107T144731_N0509_R139_T20QRF_20231107T182159.SAFE/"
                   "GRANULE/L2A_T20QRF_A043751_20231107T144729/IMG_DATA/R10m/"
                   "T20QRF_20231107T144731_B04_10m.tif"
        },
        "B08": {
            "role": "nir", "gsd_m": 10,
            "url": "https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/20/Q/RF/2023/11/07/"
                   "S2A_MSIL2A_20231107T144731_N0509_R139_T20QRF_20231107T182159.SAFE/"
                   "GRANULE/L2A_T20QRF_A043751_20231107T144729/IMG_DATA/R10m/"
                   "T20QRF_20231107T144731_B08_10m.tif"
        },
        "B11": {
            "role": "swir16", "gsd_m": 20,
            "url": "https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/20/Q/RF/2023/11/07/"
                   "S2A_MSIL2A_20231107T144731_N0509_R139_T20QRF_20231107T182159.SAFE/"
                   "GRANULE/L2A_T20QRF_A043751_20231107T144729/IMG_DATA/R20m/"
                   "T20QRF_20231107T144731_B11_20m.tif"
        },
        "SCL": {
            "role": "scene_classification", "gsd_m": 20,
            "url": "https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/20/Q/RF/2023/11/07/"
                   "S2A_MSIL2A_20231107T144731_N0509_R139_T20QRF_20231107T182159.SAFE/"
                   "GRANULE/L2A_T20QRF_A043751_20231107T144729/IMG_DATA/R20m/"
                   "T20QRF_20231107T144731_SCL_20m.tif"
        }
    },
    "worldcover": {
        "product": "ESA WorldCover 2021 v200",
        "tile": "ESA_WorldCover_10m_2021_v200_N18W063_Map.tif",
        "resolution_m": 10,
        "classes": [10,20,30,40,50,60,70,80,90,95,100],
        "url": "https://huggingface.co/datasets/cariflux/landcover-caribbean/resolve/main/"
               "ESA_WorldCover_10m_2021_v200_N18W063_Map.tif"
    },
    "copdem": {
        "product": "Copernicus DEM GLO-30",
        "type": "DSM",
        "horizontal_crs": "EPSG:4326",
        "vertical_datum": "EGM2008 / EPSG:3855",
        "resolution_arcsec": 1,
        "resolution_m_approx": 30,
        "url": "https://copernicus-dem-30m.s3.eu-central-1.amazonaws.com/"
               "Copernicus_DSM_COG_10_N18_00_W061_00_DEM/"
               "Copernicus_DSM_COG_10_N18_00_W061_00_DEM.tif"
    },
    "imerg": {
        "dataset_id": "precipitation_2024_v07",
        "units": "MM/Hr",
        "temporal_resolution_minutes": 30,
        "spatial_resolution_degree": 0.1,
        "base": "https://erddap.aoml.noaa.gov/hdb/erddap/griddap/precipitation_2024_v07"
    }
}

pd.DataFrame([
    {"source": k, **{kk:vv for kk,vv in v.items() if not isinstance(vv, dict)}}
    for k,v in SOURCES.items()
])

# Part II — Core scientific verification model

The v2 verifier uses **semantic attributes**, not filename matching, so benchmark generation and verification are not coupled by case IDs.

In [ ]:
class Verdict(str, Enum):
    PROVEN = "PROVEN"
    REFUTED = "REFUTED"
    UNPROVEN = "UNPROVEN"

@dataclass
class Artifact:
    name: str
    quantity: str
    unit: Optional[str] = None
    scale: str = "continuous"
    crs: Optional[str] = None
    resolution_m: Optional[float] = None
    temporal_resolution_h: Optional[float] = None
    processing_level: Optional[str] = None
    surface_model: Optional[str] = None
    vertical_datum: Optional[str] = None
    acquisition_time: Optional[str] = None
    available_time: Optional[str] = None
    provenance: Optional[str] = None
    band_role: Optional[str] = None
    nodata_semantics: Optional[str] = None
    uncertainty: Optional[float] = None
    attrs: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Step:
    op: str
    params: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Intent:
    goal: str
    required_unit: Optional[str] = None
    decision_time: Optional[str] = None
    max_resolution_m: Optional[float] = None
    requires_bare_earth: bool = False
    requires_cloud_mask: bool = False
    max_uncertainty: Optional[float] = None

@dataclass
class Case:
    case_id: str
    tier: str
    artifacts: List[Artifact]
    steps: List[Step]
    intent: Intent
    expected: str
    family: str
    split: str
    generation_note: str = ""

def dt(x):
    if x is None:
        return None
    z = datetime.fromisoformat(x.replace("Z","+00:00"))
    if z.tzinfo is None:
        z = z.replace(tzinfo=timezone.utc)
    return z

AREA_UNITS = {"m2","km2","ha"}
RATE_UNITS = {"mm/h","mm/hr","MM/Hr"}
ACCUM_UNITS = {"mm","cm","m"}
CATEGORICAL_SCALES = {"categorical","nominal","ordinal","binary"}
CONTINUOUS_RESAMPLERS = {"bilinear","cubic","lanczos"}

In [ ]:
def verify_case(case: Case) -> Dict[str, Any]:
    issues = []
    a = [replace(x) for x in case.artifacts]

    def add(code, verdict, msg, step=None, repair=None):
        issues.append({
            "code": code, "verdict": verdict, "message": msg,
            "step": step, "repair": repair
        })

    # Source-level checks
    for i,x in enumerate(a):
        if not x.provenance:
            add("PROV_MISSING","UNPROVEN","Source provenance is absent.", i,
                "Attach source/product/version metadata.")
        if x.resolution_m is not None and x.resolution_m <= 0:
            add("RES_INVALID","REFUTED","Non-positive spatial resolution.", i)

    for si, step in enumerate(case.steps):
        op = step.op
        p = step.params

        if op == "resample":
            idx = p.get("artifact",0)
            method = p.get("method","nearest").lower()
            target = float(p["target_resolution_m"])
            x = a[idx]
            if x.scale in CATEGORICAL_SCALES and method in CONTINUOUS_RESAMPLERS:
                add("CAT_INTERP","REFUTED",
                    f"{method} interpolation is incompatible with {x.scale} measurements.",
                    si, "Use nearest neighbour or a categorical aggregation.")
            if x.resolution_m and target < x.resolution_m:
                factor = x.resolution_m / target
                if factor >= 2 and not p.get("super_resolution_model"):
                    add("UNSUPPORTED_SUPERRES","REFUTED",
                        f"Grid spacing is increased {factor:.1f}x without a super-resolution model/evidence.",
                        si, "Do not claim newly created physical detail.")
            x.resolution_m = target

        elif op == "temporal_aggregate":
            idx = p.get("artifact",0)
            x = a[idx]
            method = p.get("method","mean")
            duration_h = p.get("sample_duration_h")
            declared = p.get("declared_quantity")
            if x.unit in RATE_UNITS and declared == "accumulation":
                if method == "mean":
                    add("RATE_MEAN_AS_ACCUM","REFUTED",
                        "A precipitation rate was averaged and declared as accumulation.", si,
                        "Integrate the rate over time.")
                elif method == "sum" and duration_h not in (1,1.0):
                    add("RATE_SUM_NO_DURATION","REFUTED",
                        "Rates were summed without multiplying by the sample duration.", si,
                        "Multiply each rate by its interval duration before summing.")
                elif method in {"integrate","duration_weighted_sum"}:
                    x.unit = "mm"
                    x.quantity = "precipitation_accumulation"

        elif op == "area":
            idx = p.get("artifact",0)
            x = a[idx]
            method = p.get("method","planar")
            out_unit = p.get("output_unit","km2")
            if x.crs in {"EPSG:4326","OGC:CRS84"} and method == "planar":
                add("ANGULAR_AREA","REFUTED",
                    "Planar area is computed directly from angular coordinates.", si,
                    "Use geodesic area or an appropriate projected/equal-area CRS.")
            x.quantity = "area"
            x.unit = out_unit

        elif op == "index":
            red_i = p["red_artifact"]
            other_i = p["other_artifact"]
            expected_other = p.get("expected_other_role")
            red = a[red_i]
            other = a[other_i]
            if red.band_role != "red":
                add("WRONG_RED_BAND","REFUTED",
                    f"Expected red band but received {red.band_role}.", si)
            if expected_other and other.band_role != expected_other:
                add("WRONG_SPECTRAL_ROLE","REFUTED",
                    f"Expected {expected_other} but received {other.band_role}.", si)
            if red.resolution_m and other.resolution_m and red.resolution_m != other.resolution_m:
                if not p.get("explicit_alignment",False):
                    add("BAND_RES_MISMATCH","REFUTED",
                        "Spectral inputs have different native resolutions without explicit alignment.", si)

        elif op == "require_cloud_clean":
            idx = p.get("artifact",0)
            x = a[idx]
            if not x.attrs.get("cloud_mask_applied",False):
                add("CLOUD_MASK_OMITTED","REFUTED",
                    "Cloud-sensitive analysis requires a cloud mask, but none is recorded.", si)

        elif op == "require_surface_reflectance":
            idx = p.get("artifact",0)
            x = a[idx]
            if x.processing_level not in {"L2A","surface_reflectance"}:
                add("WRONG_PROCESSING_LEVEL","REFUTED",
                    f"Surface reflectance required; got {x.processing_level}.", si)

        elif op == "require_bare_earth":
            idx = p.get("artifact",0)
            x = a[idx]
            if x.surface_model == "DSM":
                add("DSM_AS_DTM","REFUTED",
                    "Bare-earth analysis is requested from a DSM.", si,
                    "Use a DTM or justify the DSM for this purpose.")
            elif x.surface_model is None:
                add("SURFACE_MODEL_UNKNOWN","UNPROVEN",
                    "Cannot establish DSM/DTM semantics.", si)

        elif op == "combine_vertical":
            i,j = p["a"],p["b"]
            x,y = a[i],a[j]
            if not x.vertical_datum or not y.vertical_datum:
                add("VDATUM_UNKNOWN","UNPROVEN",
                    "Vertical-datum compatibility cannot be established.", si)
            elif x.vertical_datum != y.vertical_datum:
                add("VDATUM_MISMATCH","REFUTED",
                    f"Vertical datums differ: {x.vertical_datum} vs {y.vertical_datum}.", si)

        elif op == "predict":
            decision = dt(case.intent.decision_time)
            for ai in p.get("features", range(len(a))):
                t = dt(a[ai].available_time or a[ai].acquisition_time)
                if decision and t and t > decision:
                    add("FUTURE_LEAK","REFUTED",
                        "A feature becomes available after the prediction decision time.", si)
                if decision and t is None:
                    add("TIME_UNKNOWN","UNPROVEN",
                        "Feature availability time is missing.", si)

        elif op == "split_spatial":
            buffer_m = float(p.get("buffer_m",0))
            corr_range_m = float(p.get("estimated_autocorrelation_range_m",0))
            if corr_range_m > 0 and buffer_m < corr_range_m:
                add("SPATIAL_LEAK","REFUTED",
                    f"Train/test separation {buffer_m} m is below estimated spatial dependence range {corr_range_m} m.",
                    si)

        elif op == "fill_nodata":
            idx = p.get("artifact",0)
            x = a[idx]
            value = p.get("value")
            if value == 0 and x.nodata_semantics in {"missing","masked"} and p.get("zero_is_valid",True):
                add("NODATA_TO_ZERO","REFUTED",
                    "Missing observations are replaced by a physically valid zero value.", si)

        elif op == "multisensor_compare":
            i,j = p["a"],p["b"]
            tol_h = float(p.get("tolerance_h",0))
            ta,tb = dt(a[i].acquisition_time), dt(a[j].acquisition_time)
            if ta and tb:
                gap = abs((ta-tb).total_seconds())/3600
                if gap > tol_h:
                    add("TEMPORAL_MISMATCH","REFUTED",
                        f"Sensor acquisition gap {gap:.1f} h exceeds tolerance {tol_h:.1f} h.", si)
            else:
                add("ACQ_TIME_UNKNOWN","UNPROVEN",
                    "Multisensor acquisition-time compatibility is unknown.", si)

        elif op == "declare_uncertainty":
            idx = p.get("artifact",0)
            a[idx].uncertainty = p.get("value")

    # Intent-level checks
    if case.intent.required_unit:
        final_units = {x.unit for x in a if x.unit is not None}
        if case.intent.required_unit not in final_units:
            # only flag if a result-like quantity exists
            resultish = [x for x in a if x.quantity in {"area","precipitation_accumulation"}]
            if resultish:
                add("OUTPUT_UNIT","REFUTED",
                    f"Required unit {case.intent.required_unit} is not present in result artifacts.")

    if case.intent.max_resolution_m is not None:
        for x in a:
            if x.resolution_m and x.resolution_m > case.intent.max_resolution_m:
                add("RES_TOO_COARSE","REFUTED",
                    f"{x.name} resolution {x.resolution_m} m is too coarse for the declared {case.intent.max_resolution_m} m limit.")

    if case.intent.max_uncertainty is not None:
        known = [x.uncertainty for x in a if x.uncertainty is not None]
        if not known:
            add("UNCERTAINTY_UNKNOWN","UNPROVEN","Required uncertainty evidence is missing.")
        elif max(known) > case.intent.max_uncertainty:
            add("UNCERTAINTY_EXCEEDS","REFUTED","Declared uncertainty exceeds the permitted threshold.")

    if any(x["verdict"]=="REFUTED" for x in issues):
        verdict = "REFUTED"
    elif any(x["verdict"]=="UNPROVEN" for x in issues):
        verdict = "UNPROVEN"
    else:
        verdict = "PROVEN"

    return {"case_id":case.case_id,"verdict":verdict,"issues":issues}

# Part III — Optional live pixel experiments

When `LIVE_DATA=True`, these cells use network-accessible public sources.

They are written to use **small windows/subsets** rather than download full global products.

In [ ]:
def live_dependencies():
    if not LIVE_DATA:
        print("LIVE_DATA=False — skipping package installation.")
        return
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "rasterio", "requests", "xarray", "netCDF4", "matplotlib"
    ])
    print("Live-data dependencies ready.")

live_dependencies()

In [ ]:
def read_remote_cog_window(url, row_off=5000, col_off=5000, size=512):
    if not LIVE_DATA:
        return None
    import rasterio
    from rasterio.windows import Window
    with rasterio.open(url) as src:
        row_off = min(row_off, max(0, src.height-size))
        col_off = min(col_off, max(0, src.width-size))
        arr = src.read(1, window=Window(col_off,row_off,size,size))
        return {
            "array": arr,
            "crs": str(src.crs),
            "transform": src.window_transform(Window(col_off,row_off,size,size)),
            "dtype": str(arr.dtype),
            "shape": arr.shape,
            "nodata": src.nodata,
            "resolution": src.res,
        }

if LIVE_DATA:
    s2_red = read_remote_cog_window(SOURCES["sentinel2"]["B04"]["url"])
    s2_nir = read_remote_cog_window(SOURCES["sentinel2"]["B08"]["url"])
    print("B04:", s2_red["shape"], s2_red["resolution"], s2_red["crs"])
    print("B08:", s2_nir["shape"], s2_nir["resolution"], s2_nir["crs"])
else:
    print("Sentinel-2 pixel experiment ready; enable LIVE_DATA in Colab to execute the remote COG read.")

In [ ]:
if LIVE_DATA:
    red = s2_red["array"].astype("float32")
    nir = s2_nir["array"].astype("float32")
    denom = nir + red
    ndvi = np.divide(nir-red, denom, out=np.full_like(red,np.nan), where=denom!=0)

    ndvi_stats = {
        "min": float(np.nanmin(ndvi)),
        "median": float(np.nanmedian(ndvi)),
        "max": float(np.nanmax(ndvi)),
        "finite_fraction": float(np.isfinite(ndvi).mean())
    }
    print(ndvi_stats)
else:
    # deterministic fixture only for offline notebook execution
    red = np.array([[1000,1500],[2000,2500]], dtype=float)
    nir = np.array([[3000,2700],[2200,2100]], dtype=float)
    ndvi = (nir-red)/(nir+red)
    print("Offline NDVI fixture:", ndvi)

In [ ]:
def worldcover_resampling_experiment():
    if not LIVE_DATA:
        original = np.array([[10,10,40],[10,40,40],[80,80,40]], dtype=np.float32)
        # simple midpoint-style demonstration of continuous interpolation creating invalid categories
        bilinear_like = np.array([[10,25],[45,55]], dtype=np.float32)
        nearest_like = np.array([[10,40],[80,40]], dtype=np.float32)
        return {
            "mode":"offline_fixture",
            "original_unique": sorted(np.unique(original).tolist()),
            "nearest_unique": sorted(np.unique(nearest_like).tolist()),
            "bilinear_unique": sorted(np.unique(bilinear_like).tolist()),
            "invalid_bilinear_values": sorted(
                set(np.unique(bilinear_like).tolist()) - set(SOURCES["worldcover"]["classes"])
            )
        }

    import requests, tempfile, rasterio
    from rasterio.enums import Resampling
    url = SOURCES["worldcover"]["url"]
    path = Path("/content/worldcover_real.tif")
    if not path.exists():
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        path.write_bytes(r.content)

    with rasterio.open(path) as src:
        w = min(768, src.width)
        h = min(768, src.height)
        src_arr = src.read(1, window=rasterio.windows.Window(0,0,w,h))

        out_h = max(1,h//3); out_w=max(1,w//3)
        nn = src.read(1, out_shape=(out_h,out_w), resampling=Resampling.nearest)
        bl = src.read(1, out_shape=(out_h,out_w), resampling=Resampling.bilinear)

    allowed = set(SOURCES["worldcover"]["classes"])
    return {
        "mode":"live_real_worldcover",
        "original_unique": sorted(np.unique(src_arr).tolist()),
        "nearest_unique": sorted(np.unique(nn).tolist()),
        "bilinear_unique_sample": sorted(np.unique(bl).tolist())[:50],
        "invalid_bilinear_values": sorted(set(np.unique(bl).tolist()) - allowed)[:50],
    }

wc_experiment = worldcover_resampling_experiment()
wc_experiment

In [ ]:
def imerg_query_url(
    start="2024-07-01T00:00:00Z",
    stop="2024-07-01T23:30:00Z",
    lat_min=18.25, lat_max=18.55,
    lon_min=-60.20, lon_max=-60.00
):
    base = SOURCES["imerg"]["base"]
    # CSV is easy to inspect and avoids Earthdata authentication.
    q = (
        f"precipitation[({start}):1:({stop})]"
        f"[({lat_min}):1:({lat_max})]"
        f"[({lon_min}):1:({lon_max})]"
    )
    return base + ".csv?" + q

print(imerg_query_url())

In [ ]:
def imerg_real_subset_experiment():
    if not LIVE_DATA:
        # 48 half-hourly rate samples
        rates = np.array(
            [0]*8 + [0.5,1,2,4,8,12,10,6,4,2,1,0] + [0]*12 +
            [0.5,1,2,3,5,4,2,1] + [0]*8,
            dtype=float
        )[:48]
        correct = float(np.sum(rates * 0.5))
        wrong_mean = float(np.mean(rates))
        wrong_sum = float(np.sum(rates))
        return {
            "mode":"offline_fixture",
            "correct_accumulation_mm":correct,
            "wrong_mean_labelled_mm":wrong_mean,
            "wrong_raw_sum_labelled_mm":wrong_sum
        }

    import pandas as pd
    url = imerg_query_url()
    df = pd.read_csv(url)
    # ERDDAP response has one row per time/lat/lon point.
    vals = pd.to_numeric(df["precipitation (MM/Hr)"], errors="coerce").dropna()
    return {
        "mode":"live_real_imerg",
        "n_values":int(len(vals)),
        "correct_integrated_mm_per_gridpoint_mean":float((vals*0.5).groupby(
            [df.loc[vals.index,"latitude (degrees_north)"],
             df.loc[vals.index,"longitude (degrees_east)"]]
        ).sum().mean()),
        "naive_rate_mean":float(vals.mean())
    }

imerg_experiment = imerg_real_subset_experiment()
imerg_experiment

In [ ]:
def copdem_live_metadata():
    if not LIVE_DATA:
        return {
            "mode":"offline_specification",
            "surface_model":SOURCES["copdem"]["type"],
            "horizontal_crs":SOURCES["copdem"]["horizontal_crs"],
            "vertical_datum":SOURCES["copdem"]["vertical_datum"],
            "resolution_m_approx":SOURCES["copdem"]["resolution_m_approx"]
        }
    import rasterio
    with rasterio.open(SOURCES["copdem"]["url"]) as src:
        arr = src.read(1, window=rasterio.windows.Window(300,300,256,256))
        return {
            "mode":"live_real_copdem",
            "crs":str(src.crs),
            "resolution":src.res,
            "min":float(np.nanmin(arr)),
            "median":float(np.nanmedian(arr)),
            "max":float(np.nanmax(arr)),
        }

copdem_live_metadata()

# Part IV — SilentGeoBench v2

Unlike the first notebook, v2 introduces:

- multiple parameter combinations per error family;
- atomic, compositional, and adversarial tiers;
- development and **held-out family** test splits;
- valid controls;
- case hashes to detect duplicates.

This remains an automatically generated benchmark and should be expert-reviewed before publication.

In [ ]:
DEV_FAMILIES = [
    "categorical_interpolation",
    "rate_mean_as_accum",
    "angular_area",
    "band_resolution_mismatch",
    "cloud_mask_omission",
    "wrong_processing_level",
    "DSM_as_DTM",
    "future_temporal_leak",
    "spatial_leakage",
    "nodata_zero_fill",
    "unsupported_superresolution",
    "multisensor_temporal_mismatch",
]

HELDOUT_FAMILIES = [
    "rate_sum_without_duration",
    "wrong_spectral_role",
    "vertical_datum_mismatch",
    "missing_provenance",
    "unknown_vertical_datum",
    "uncertainty_missing",
]

ALL_FAMILIES = DEV_FAMILIES + HELDOUT_FAMILIES
len(ALL_FAMILIES), len(DEV_FAMILIES), len(HELDOUT_FAMILIES)

In [ ]:
def make_artifact(**kw):
    defaults = dict(
        name="artifact", quantity="unknown", unit=None, scale="continuous",
        crs="EPSG:32620", resolution_m=10, provenance="authoritative-source"
    )
    defaults.update(kw)
    return Artifact(**defaults)

def build_case(family, idx, split):
    rng = random.Random(SEED*10000 + idx*31 + sum(map(ord,family)))
    tier = rng.choice(["atomic","compositional","adversarial"])
    good = rng.random() < 0.25

    # family-specific valid/invalid parameterization
    if family == "categorical_interpolation":
        art = make_artifact(name=f"WorldCover-{idx}", quantity="land_cover",
                            scale="categorical", crs="EPSG:4326",
                            resolution_m=rng.choice([10,20,30]))
        method = rng.choice(["nearest","mode"]) if good else rng.choice(["bilinear","cubic"])
        steps=[Step("resample",{"artifact":0,"method":method,
                                "target_resolution_m":rng.choice([20,30,60])})]

    elif family == "rate_mean_as_accum":
        art=make_artifact(name=f"IMERG-{idx}",quantity="precipitation_rate",
                          unit="mm/h",crs="EPSG:4326",resolution_m=10000,
                          temporal_resolution_h=0.5)
        method="integrate" if good else "mean"
        steps=[Step("temporal_aggregate",{"artifact":0,"method":method,
                                          "sample_duration_h":0.5,
                                          "declared_quantity":"accumulation"})]

    elif family == "rate_sum_without_duration":
        art=make_artifact(name=f"IMERG-{idx}",quantity="precipitation_rate",
                          unit="mm/h",crs="EPSG:4326",resolution_m=10000,
                          temporal_resolution_h=0.5)
        method="integrate" if good else "sum"
        steps=[Step("temporal_aggregate",{"artifact":0,"method":method,
                                          "sample_duration_h":0.5,
                                          "declared_quantity":"accumulation"})]

    elif family == "angular_area":
        art=make_artifact(name=f"FloodMask-{idx}",quantity="flood_extent",
                          unit=None,scale="binary",
                          crs=("EPSG:32620" if good else "EPSG:4326"))
        steps=[Step("area",{"artifact":0,"method":"planar","output_unit":"km2"})]

    elif family == "band_resolution_mismatch":
        red=make_artifact(name="B04",quantity="reflectance",processing_level="L2A",
                          band_role="red",resolution_m=10)
        other=make_artifact(name=("B08" if good else "B11"),quantity="reflectance",
                            processing_level="L2A",band_role=("nir" if good else "swir16"),
                            resolution_m=(10 if good else 20))
        steps=[Step("index",{"red_artifact":0,"other_artifact":1,
                             "expected_other_role":other.band_role if not good else "nir",
                             "explicit_alignment":good})]
        return Case("",tier,[red,other],steps,Intent("multiband index"),
                    "PROVEN" if good else "REFUTED",family,split)

    elif family == "wrong_spectral_role":
        red=make_artifact(name="B04",quantity="reflectance",processing_level="L2A",
                          band_role="red",resolution_m=10)
        other=make_artifact(name=("B08" if good else "B11"),quantity="reflectance",
                            processing_level="L2A",band_role=("nir" if good else "swir16"),
                            resolution_m=(10 if good else 20))
        steps=[Step("index",{"red_artifact":0,"other_artifact":1,
                             "expected_other_role":"nir","explicit_alignment":True})]
        return Case("",tier,[red,other],steps,Intent("NDVI"),
                    "PROVEN" if good else "REFUTED",family,split)

    elif family == "cloud_mask_omission":
        art=make_artifact(name="S2",quantity="surface_reflectance",
                          processing_level="L2A",band_role="red",
                          attrs={"cloud_mask_applied":good})
        steps=[Step("require_cloud_clean",{"artifact":0})]

    elif family == "wrong_processing_level":
        art=make_artifact(name="S2",quantity="reflectance",
                          processing_level=("L2A" if good else rng.choice(["L1C","TOA"])),
                          band_role="red")
        steps=[Step("require_surface_reflectance",{"artifact":0})]

    elif family == "DSM_as_DTM":
        art=make_artifact(name="DEM",quantity="elevation",
                          surface_model=("DTM" if good else "DSM"),
                          vertical_datum="EGM2008 / EPSG:3855",unit="m",resolution_m=30)
        steps=[Step("require_bare_earth",{"artifact":0})]

    elif family == "vertical_datum_mismatch":
        a=make_artifact(name="DEM",quantity="elevation",unit="m",vertical_datum="EGM2008")
        b=make_artifact(name="WaterLevel",quantity="water_level",unit="m",
                        vertical_datum=("EGM2008" if good else "ellipsoidal_WGS84"))
        steps=[Step("combine_vertical",{"a":0,"b":1})]
        return Case("",tier,[a,b],steps,Intent("relative water depth"),
                    "PROVEN" if good else "REFUTED",family,split)

    elif family == "unknown_vertical_datum":
        a=make_artifact(name="DEM",quantity="elevation",unit="m",vertical_datum="EGM2008")
        b=make_artifact(name="WaterLevel",quantity="water_level",unit="m",
                        vertical_datum=("EGM2008" if good else None))
        steps=[Step("combine_vertical",{"a":0,"b":1})]
        return Case("",tier,[a,b],steps,Intent("relative water depth"),
                    "PROVEN" if good else "UNPROVEN",family,split)

    elif family == "future_temporal_leak":
        decision=datetime(2024,7,1,12,tzinfo=timezone.utc)
        offset_h=rng.choice([-12,-6,-1,1,6,12])
        if good and offset_h>0: offset_h=-offset_h
        if (not good) and offset_h<0: offset_h=-offset_h
        art=make_artifact(name="rain",quantity="rainfall",unit="mm",
                          acquisition_time=(decision+timedelta(hours=offset_h)).isoformat())
        steps=[Step("predict",{"features":[0]})]
        intent=Intent("flood prediction",decision_time=decision.isoformat())
        return Case("",tier,[art],steps,intent,"PROVEN" if good else "REFUTED",family,split)

    elif family == "spatial_leakage":
        corr=rng.choice([500,1000,2000,5000])
        buffer=(corr+rng.choice([500,1000])) if good else rng.choice([0,corr/4,corr/2])
        art=make_artifact(name="samples",quantity="training_samples")
        steps=[Step("split_spatial",{"buffer_m":buffer,
                                     "estimated_autocorrelation_range_m":corr})]

    elif family == "nodata_zero_fill":
        art=make_artifact(name="rain",quantity="rainfall",unit="mm",
                          nodata_semantics="missing")
        steps=[Step("fill_nodata",{"artifact":0,
                                   "value":(-9999 if good else 0),
                                   "zero_is_valid":True})]

    elif family == "unsupported_superresolution":
        native=rng.choice([20,30,100])
        target=(native*2 if good else native/rng.choice([2,3,4]))
        art=make_artifact(name="raster",quantity="surface_property",
                          resolution_m=native)
        steps=[Step("resample",{"artifact":0,"method":"bilinear",
                                "target_resolution_m":target,
                                "super_resolution_model": good and target<native})]

    elif family == "multisensor_temporal_mismatch":
        t0=datetime(2024,7,1,10,tzinfo=timezone.utc)
        gap_h=rng.choice([0,6,12,24,48])
        tol=rng.choice([6,12,24])
        if good:
            gap_h=min(gap_h,tol)
        else:
            gap_h=max(gap_h,tol+6)
        a=make_artifact(name="S1",quantity="backscatter",acquisition_time=t0.isoformat())
        b=make_artifact(name="S2",quantity="reflectance",
                        acquisition_time=(t0+timedelta(hours=gap_h)).isoformat())
        steps=[Step("multisensor_compare",{"a":0,"b":1,"tolerance_h":tol})]
        return Case("",tier,[a,b],steps,Intent("multisensor flood mapping"),
                    "PROVEN" if good else "REFUTED",family,split)

    elif family == "missing_provenance":
        art=make_artifact(name="unknown-data",quantity="elevation",
                          provenance=("official-product" if good else None))
        steps=[]

    elif family == "uncertainty_missing":
        art=make_artifact(name="hazard-map",quantity="hazard_probability",
                          uncertainty=(0.02 if good else None))
        steps=[]
        intent=Intent("decision support",max_uncertainty=0.05)
        return Case("",tier,[art],steps,intent,
                    "PROVEN" if good else "UNPROVEN",family,split)

    else:
        raise ValueError(family)

    expected="PROVEN" if good else "REFUTED"
    if family=="missing_provenance" and not good:
        expected="UNPROVEN"

    return Case("",tier,[art],steps,Intent(family),expected,family,split)


def create_benchmark(n_per_family=24):
    cases=[]
    for family in ALL_FAMILIES:
        split="dev" if family in DEV_FAMILIES else "heldout"
        for i in range(n_per_family):
            c=build_case(family,i,split)
            c.case_id=f"SGX2_{split.upper()}_{family}_{i:03d}"
            c.generation_note="parameter-diverse generated stress test; not expert-reviewed"
            cases.append(c)
    return cases

cases=create_benchmark(24)
len(cases)

In [ ]:
def case_signature(c: Case):
    payload={
        "artifacts":[asdict(x) for x in c.artifacts],
        "steps":[asdict(x) for x in c.steps],
        "intent":asdict(c.intent),
        "expected":c.expected,
        "family":c.family
    }
    return hashlib.sha256(json.dumps(payload,sort_keys=True,default=str).encode()).hexdigest()

sig=[case_signature(c) for c in cases]
print("Cases:",len(cases))
print("Unique scientific configurations:",len(set(sig)))
print("Duplicate configurations:",len(sig)-len(set(sig)))

pd.Series([c.family for c in cases]).value_counts().sort_index()

## Baselines

We intentionally include weak but interpretable baselines:

- **Runtime baseline**: everything executable is treated as valid.
- **Metadata-presence baseline**: only missing provenance/unknown metadata is flagged.
- **SciGeoGuard-X semantic verifier**: uses scientific relations and goal constraints.

The notebook does not fabricate an LLM baseline result. A separate optional cell can be added later for an actual open-source model.

In [ ]:
def baseline_runtime(c):
    return "PROVEN"

def baseline_metadata_presence(c):
    if any(x.provenance is None for x in c.artifacts):
        return "UNPROVEN"
    if any(x.vertical_datum is None for x in c.artifacts) and c.family=="unknown_vertical_datum":
        return "UNPROVEN"
    return "PROVEN"

rows=[]
for c in cases:
    pred=verify_case(c)["verdict"]
    rows.append({
        "case_id":c.case_id,
        "split":c.split,
        "tier":c.tier,
        "family":c.family,
        "expected":c.expected,
        "runtime":baseline_runtime(c),
        "metadata_baseline":baseline_metadata_presence(c),
        "scigeoguard_x":pred,
        "signature":case_signature(c)
    })

results=pd.DataFrame(rows)
results.head()

In [ ]:
def metrics(df, col):
    exp=df["expected"]
    pred=df[col]
    invalid=exp.eq("REFUTED")
    valid=exp.eq("PROVEN")

    exact=float((exp==pred).mean())
    fscr=float((pred[invalid]=="PROVEN").mean()) if invalid.any() else np.nan
    proof_coverage=float(pred.isin(["PROVEN","REFUTED"]).mean())
    valid_cert=float((pred[valid]=="PROVEN").mean()) if valid.any() else np.nan

    return {
        "accuracy":exact,
        "FSCR":fscr,
        "proof_coverage":proof_coverage,
        "valid_certification_rate":valid_cert
    }

summary=[]
for split in ["dev","heldout","all"]:
    d=results if split=="all" else results[results.split==split]
    for col in ["runtime","metadata_baseline","scigeoguard_x"]:
        summary.append({"split":split,"system":col,**metrics(d,col)})

summary_df=pd.DataFrame(summary)
summary_df

In [ ]:
family_perf = (
    results.assign(correct=lambda d: d.expected.eq(d.scigeoguard_x))
           .groupby(["split","family"])
           .agg(n=("case_id","size"), accuracy=("correct","mean"))
           .reset_index()
           .sort_values(["split","family"])
)
family_perf

## Critical interpretation

A high score on this generated benchmark is **not sufficient evidence of publication-level generalization**, because the error families are still programmatically defined.

The publication benchmark must add:

1. expert-authored workflows,
2. independently reviewed mutations,
3. real historical analyst/agent failures,
4. blind held-out cases written after the verifier is frozen.

This notebook makes that separation explicit instead of overstating generated benchmark performance.

# Part V — Expert-review worksheet generator

This export creates a review table that domain experts can annotate without changing verifier code.

In [ ]:
review_rows=[]
for c in cases:
    vr=verify_case(c)
    review_rows.append({
        "case_id":c.case_id,
        "split":c.split,
        "tier":c.tier,
        "family":c.family,
        "machine_expected":c.expected,
        "machine_verdict":vr["verdict"],
        "expert_validity":"",
        "expert_error_family":"",
        "expert_severity":"",
        "expert_comments":"",
        "adjudicated_validity":"",
        "reviewer_1":"",
        "reviewer_2":"",
    })

review_df=pd.DataFrame(review_rows)
review_df.head()

# Part VI — Publication exports

In [ ]:
export_dir=Path("/mnt/data") if Path("/mnt/data").exists() else Path.cwd()
export_dir.mkdir(parents=True,exist_ok=True)

bench_path=export_dir/"SilentGeoBench_v2_generated.csv"
results_path=export_dir/"SciGeoGuard_X_v2_results.csv"
summary_path=export_dir/"SciGeoGuard_X_v2_metrics.csv"
family_path=export_dir/"SciGeoGuard_X_v2_family_performance.csv"
review_path=export_dir/"SilentGeoBench_v2_expert_review_template.csv"
source_path=export_dir/"SciGeoGuard_X_v2_sources.json"

# full benchmark fields
bench_rows=[]
for c in cases:
    bench_rows.append({
        "case_id":c.case_id,
        "split":c.split,
        "tier":c.tier,
        "family":c.family,
        "expected":c.expected,
        "intent":json.dumps(asdict(c.intent),default=str),
        "artifacts":json.dumps([asdict(x) for x in c.artifacts],default=str),
        "steps":json.dumps([asdict(x) for x in c.steps],default=str),
        "signature":case_signature(c),
        "generation_note":c.generation_note
    })

pd.DataFrame(bench_rows).to_csv(bench_path,index=False)
results.to_csv(results_path,index=False)
summary_df.to_csv(summary_path,index=False)
family_perf.to_csv(family_path,index=False)
review_df.to_csv(review_path,index=False)
source_path.write_text(json.dumps(SOURCES,indent=2),encoding="utf-8")

for p in [bench_path,results_path,summary_path,family_path,review_path,source_path]:
    print(p)

# Part VII — What this notebook adds over v1

### v1
- proof-of-concept verifier;
- 3 main grounded benchmark families;
- repeated template instances;
- metadata-focused real-product checks.

### v2
- live remote pixel/window hooks;
- real Sentinel-2 URLs;
- real WorldCover tile access;
- real Copernicus DEM public COG endpoint;
- NOAA-hosted NASA IMERG V07 subset access;
- 18 scientific error families;
- atomic/compositional/adversarial tiers;
- dev vs held-out family split;
- duplicate detection by scientific signature;
- stronger baselines;
- FSCR and proof coverage;
- expert-review worksheet.

### Still required before a final paper
- expert review of 300–500 cases;
- blind post-freeze benchmark;
- an actual open-source LLM baseline;
- real pixel-level experiments for all four source families;
- inter-rater agreement if expert annotation is used.